<a href="https://colab.research.google.com/github/alejitovm97-byte/Alejo-Varelas-projects/blob/gh-pages/08_fases5_cartera.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =====================================================================
# 08 — FASE 5: CARTERA MULTI-ACTIVO CON RELOJ REAL-TIME
# =====================================================================
# Motor de anticipacion macro. Dos ejes calibrados a probabilidad
# alimentan un arbol de tres niveles: clases de activo, sectores dentro
# de renta variable, y nombres dentro de cada sector.
#
# ARQUITECTURA (ver fase5_cartera_integrada.md §2):
#   ESTIMACION    solo por EJE. Las fases son algebraicamente
#                 redundantes salvo por la restriccion de no
#                 negatividad.
#   CONSTRUCCION  por esquina = modelo de ejes + recorte a cero.
#   NARRATIVA     ejes. Acierto de eje 70-71%; de fase, 53%.
#
# AUTOCONTENIDO. Solo necesita:
#   - Drive montado
#   - FRED_API_KEY en .env o Colab Secrets
#   - data/raw/b1_releases_{PPIACO,CPIAUCNS}.parquet
#   - data/proc/b7_clock_expost.parquet   (salida del 06)
#
# Todo lo demas se reconstruye aca. El bloque 0 contiene funciones que
# se habian perdido y se recuperaron por identificacion contra
# resultados medidos — ver su encabezado.
#
# ESTADO: bloques 0-9 corridos y validados por separado. El archivo NO
# fue ejecutado todavia de arriba a abajo como unidad.
# =====================================================================

import io
import os
import sys
import requests
import numpy as np
import pandas as pd
from pandas.tseries.offsets import MonthEnd

BASE = '/content/drive/MyDrive/TFM_regime_allocation'
V2   = BASE + '/v2'
RAW  = V2 + '/data/raw/'
PROC = V2 + '/data/proc/'

# Drive tiene que estar montado ANTES de crear nada. Sin este guard,
# os.makedirs() crea carpetas locales vacias sobre el punto de montaje
# y el error aparece despues como "archivo no encontrado", mandando a
# buscar el problema al lugar equivocado.
if not os.path.isdir(BASE):
    raise RuntimeError(
        'Drive no montado o BASE inexistente. Correr primero:\n'
        '  from google.colab import drive\n'
        '  drive.mount("/content/drive")')
os.makedirs(RAW, exist_ok=True)
os.makedirs(PROC, exist_ok=True)

SEED = 20260913
LAG  = 12      # rezago de la verdad del reloj ex-post (D-30)


In [ ]:
# =====================================================================
# BLOQUE 0 — UTILIDADES  (recuperadas por identificacion, 2026-09-13)
# =====================================================================
# a_mensual() y merrill() existieron solo en celdas de Colab que nunca
# se guardaron. Un reinicio de kernel las borro y no estaban en ningun
# archivo ni en el .ipynb de Drive.
#
# SE RECUPERARON POR IDENTIFICACION contra resultados ya medidos, no
# por memoria. Cada una tiene su prueba:
#
# merrill — por eliminacion (D-29). Hay tres formas plausibles:
#   x − MA(x), log(x) − log(MA(x)), y log(x) − media(log x). Las dos
#   primeras son INVARIANTES a sumarle una constante a la serie; la
#   tercera no. Medimos que el desplazamiento de ICSA cambiaba entre
#   0.25% y 1.6% de los signos. Solo la tercera es compatible.
#
# oil = WTISPLC — reproduce 66.5% (MA6) y 70.6% (MA12) EXACTOS sobre
#   418 meses. DCOILWTICO y MCOILWTICO dan 66.7/70.8: cerca, pero no.
#
# a_mensual = resample('ME').last() — reproduce los CINCO numeros del
#   comite de crecimiento: n=335, sin centrar 71.3%/47.8%, centrado
#   59.7%/30.1%. Con .mean() da 73.7%/48.4% y 60.9%/31.9%.
#   Nota: .mean() rendia MEJOR que .last() (73.7% vs 71.3%), asi que la
#   eleccion original no fue por resultado. Solo importa para ICSA, que
#   es semanal; las otras tres series ya son mensuales.
#
# LECCION: nada critico puede vivir solo en un notebook. Este bloque
# existe para que el 08 corra desde un kernel limpio.

from pathlib import Path
from fredapi import Fred


def get_secret(nombre):
    """Secreto por nombre. Nunca lo imprime."""
    try:
        from google.colab import userdata
        v = userdata.get(nombre)
        if v:
            return v
    except Exception:
        pass
    v = os.environ.get(nombre)
    if v:
        return v
    for p in [Path(V2) / '.env', Path(BASE) / '.env']:
        if p.exists():
            for ln in p.read_text().splitlines():
                ln = ln.strip()
                if ln.startswith(nombre + '='):
                    return ln.split('=', 1)[1].strip().strip('"\'')
    raise KeyError(f'falta el secreto {nombre}')


def a_mensual(x):
    """Serie FRED a mensual, ultimo dato del mes."""
    x = pd.Series(x).dropna()
    x.index = pd.to_datetime(x.index)
    return x.resample('ME').last().dropna()


def merrill(s, n):
    """log(s) menos la media movil de los LOGARITMOS.
    Media geometrica: NO es invariante a desplazar la serie."""
    ls = np.log(s)
    return ls - ls.rolling(n).mean()


fred = Fred(api_key=get_secret('FRED_API_KEY'))

CRE = ['ICSA', 'PERMIT', 'AWHMAN', 'NEWORDER']
SGN = {'ICSA': -1, 'PERMIT': 1, 'AWHMAN': 1, 'NEWORDER': 1}

dat = pd.DataFrame({k: a_mensual(fred.get_series(k))
                    for k in CRE + ['T10Y2Y']})
oil = a_mensual(fred.get_series('WTISPLC'))

cx = pd.read_parquet(PROC + 'b7_clock_expost.parquet')
cx.index = pd.to_datetime(cx.index) + MonthEnd(0)


def serie_pit(rel, asof):
    """Serie tal como se conocia en la fecha asof."""
    x = rel[rel.realtime_start <= asof]
    return (x.sort_values('realtime_start')
             .groupby('date')['value'].last()
             .sort_index())


def pit_rapido(rel, idx):
    """Eje rapido de inflacion, reconstruido sin look-ahead.
    Formula literal de Merrill: yoy menos su media movil de 12.
    El ffill() antes de pct_change replica el default 'pad' que pandas
    esta deprecando; se deja explicito porque los numeros verificados
    salieron con ese comportamiento."""
    r0 = rel.realtime_start.min()
    out = {}
    for f in idx:
        fm = f + MonthEnd(0)
        s = serie_pit(rel, max(fm, r0))
        s = s[s.index <= fm]
        yoy = s.ffill().pct_change(12)
        v = (yoy - yoy.rolling(12).mean()).dropna()
        if len(v):
            out[f] = v.iloc[-1]
    return pd.Series(out)


# VERIFICADO: 62.7% de acierto, dice sube 48.1%, n=418 sobre
# 1991-01 a 2025-10. Los tres numeros exactos contra la version
# original perdida.
cpi_rel = pd.read_parquet(RAW + 'b1_releases_CPIAUCNS.parquet')
ir = pit_rapido(cpi_rel,
                pd.date_range('1985-01-31', '2025-12-31', freq='ME'))


In [ ]:
# =====================================================================
# BLOQUE 1 — PPI POINT-IN-TIME  (D-28)
# =====================================================================
# Ultima deuda de vintages. El PPI entraba con datos finales, lo que
# daba look-ahead de un mes entero (el dato del mes m no se publica
# hasta mediados de m+1).
#
# Revisiones medidas: mediana 0.10% del nivel, p99 0.83%. Sobre la
# senal, correlacion 0.9969 y acuerdo de signo 97.99%. Pero el efecto
# TOTAL incluye el retraso de publicacion, que es lo que muerde:
# ppi_MA6 pasa de 72.0% a 69.4% de acierto.
#
# Vintages solo desde 1996-12-11. Para los 71 meses previos se usa el
# vintage de dic-96 con lag fijo de 1 mes, marcado con es_vintage.
#
# HALLAZGO COLATERAL: el cierre del gobierno de EE.UU. de oct-nov 2025
# retraso publicaciones hasta 3 meses (obs de oct-2025 publicada el
# 2026-01-14). Un supuesto de lag fijo no ve el apagon.

ppi_rel = pd.read_parquet(RAW + 'b1_releases_PPIACO.parquet')


def pit_merrill(rel, n, idx, lag_fb=1):
    """merrill(s, n) reconstruido mes a mes sin look-ahead.
    Dos ramas: con vintage real el filtro por realtime_start hace
    todo el trabajo; sin vintage, se usa el unico disponible mas un
    lag fijo de publicacion."""
    r0 = rel.realtime_start.min()
    out = {}
    for f in idx:
        fm = f + MonthEnd(0)
        if fm >= r0:
            asof, tope = fm, fm
        else:
            asof, tope = r0, fm - MonthEnd(lag_fb)
        s = serie_pit(rel, asof)
        s = s[s.index <= tope]
        v = merrill(s, n).dropna()
        if len(v):
            out[f] = v.iloc[-1]
    return pd.Series(out)


ppi   = a_mensual(fred.get_series('PPIACO'))
ref6  = merrill(ppi, 6).dropna()
ppi6_pit  = pit_merrill(ppi_rel, 6,  ref6.index)
ppi12_pit = pit_merrill(ppi_rel, 12, ref6.index)

PPI_PIT = pd.DataFrame({'ppi_pit': ppi6_pit,
                        'ppi12_pit': ppi12_pit})
PPI_PIT['es_vintage'] = (PPI_PIT.index
                         >= ppi_rel.realtime_start.min())
PPI_PIT.to_parquet(RAW + 'b1_ppi_pit.parquet')
# validado: (962, 3), 357 meses con vintage real


In [ ]:
# =====================================================================
# BLOQUE 2 — COMITE DE INFLACION
# =====================================================================
# Tres fuentes de baja correlacion entre si (0.21 a 0.56). La
# composicion se fijo con criterios pre-registrados en Fase 5 y NO se
# revisa aunque los numeros se muevan.
#
# Resultado con PPI point-in-time, 418 meses 1991-01 a 2025-10:
#   oil_MA6     66.5% | oil_MA12   70.6% | cpi_rapido 62.7%
#   ppi_MA6     69.4% | ppi_MA12   62.2%
#   comite de los tres MA6: 70.8%   (benchmark 50.5%)
# Costo de sacar el look-ahead del PPI: -1.0 pp (era 71.8%).

cand_i = {
    'oil_MA6':    merrill(oil, 6),
    'oil_MA12':   merrill(oil, 12),
    'cpi_rapido': ir,
    'ppi_MA6':    ppi6_pit,
    'ppi_MA12':   ppi12_pit,
}
TRES_I = ['oil_MA6', 'cpi_rapido', 'ppi_MA6']


In [ ]:
# =====================================================================
# BLOQUE 3 — COMITE DE CRECIMIENTO  (D-29)
# =====================================================================
# CORRECCION: la version vieja hacia merrill(s - s.min() + 1, 12) para
# poder logaritmar la serie invertida. s.min() es el minimo de la
# MUESTRA COMPLETA, y para ICSA invertido queda dominado por el pico de
# COVID de abril 2020 aplicado retroactivamente a toda la historia.
#
# Se creyo primero que era inocuo por invarianza del signo al
# desplazamiento. ES FALSO: merrill usa la media de los LOGARITMOS
# (geometrica), no del nivel. Medido: el signo cambia en 0.25% a 1.6%
# de los meses.
#
# ARREGLO: transformar primero e invertir despues. Las cuatro series
# son positivas, asi que no hace falta desplazamiento.
#
# Efecto colateral que importaba: el desplazamiento comprimia la
# MAGNITUD de ICSA unas 10 veces. Irrelevante mientras solo se usaba el
# signo; fatal al pasar a estandarizar y calibrar.

CRE = ['ICSA', 'PERMIT', 'AWHMAN', 'NEWORDER']
SGN = {'ICSA': -1, 'PERMIT': 1, 'AWHMAN': 1, 'NEWORDER': 1}

sig = {k: (SGN[k] * merrill(dat[k], 12)).dropna()
       for k in CRE}
# escalas post-arreglo: ICSA 0.204, PERMIT 0.118,
#                       NEWORDER 0.048, AWHMAN 0.012


In [ ]:
# =====================================================================
# BLOQUE 4 — CALIBRACION DE LOS EJES  (D-30)
# =====================================================================
# La cartera dimensiona con P(eje sube), asi que necesita una
# PROBABILIDAD calibrada, no un signo.
#
# CAMINO FALLIDO: centrar en la mediana expansiva. Inflacion 71.9% ->
# 74.7% (adoptado), crecimiento 71.3% -> 59.7% (rechazado). Centrar =
# poner el umbral en el percentil 50, y el percentil correcto lo fija
# la TASA BASE: inflacion sube ~52% (percentil 50 casi correcto, acerto
# por coincidencia), crecimiento sube 61-65% (deberia estar cerca del
# percentil 37).
#
# SOLUCION: un procedimiento para los dos ejes.
#   1. estandarizar cada senal en ventana expansiva
#   2. promediar EXIGIENDO QUE ESTEN TODAS (mean() de pandas saltea
#      NaN, y sin esto el comite cambia de composicion a mitad de
#      muestra: la primera corrida reportaba 785 meses desde 1960
#      promediando dos senales donde no habia cuatro)
#   3. logistica de 2 parametros, refit mensual expansivo, verdad
#      rezagada LAG meses
#
# Resultado, 288 meses comunes 2001-11 a 2025-10:
#   crecimiento  Brier 0.1878 vs 0.2503 base  -> +25.0%
#   inflacion    Brier 0.1903 vs 0.2631 base  -> +27.7%
# Adoptado por la clausula 3 del criterio pre-registrado: el acierto de
# crecimiento cayo 1.4 pp (excede la clausula 2) pero el Brier mejoro
# en los dos ejes y la cartera consume la probabilidad, no el signo.
#
# LIMITACION SIN ARREGLO: P(crec) promedia 0.726 contra una verdad de
# 0.646. La logistica expansiva aprende la tasa base del pasado y la
# tasa base se movio (70% en 2008-25, 65% en 2001-25, 61% largo plazo).
# Es el precio de no tener look-ahead.

from sklearn.linear_model import LogisticRegression

MINP   = 36    # burn-in de estandarizacion (media y sd: ~30 obs)
MINFIT = 60    # minimo para calibrar (logistica de 2 parametros)


def zexp(s, minp=MINP):
    m  = s.expanding(min_periods=minp).mean()
    sd = s.expanding(min_periods=minp).std()
    return (s - m) / sd


def hacer_score(sigd, keys):
    Z = pd.DataFrame({k: zexp(sigd[k].dropna()) for k in keys})
    return Z.dropna().mean(1)        # dropna ANTES de promediar


def calibrar(sc, verdad, lag=LAG, minfit=MINFIT):
    ix = sc.index
    p_cal, p_base = {}, {}
    for i, t in enumerate(ix):
        if i < lag:
            continue
        x = sc.loc[:ix[i - lag]]
        y = verdad.reindex(x.index).dropna()
        x = x.reindex(y.index)
        if len(y) < minfit or y.nunique() < 2:
            continue
        lr = LogisticRegression(C=1e6)
        lr.fit(x.values.reshape(-1, 1), y.astype(int).values)
        p_cal[t]  = lr.predict_proba([[sc.loc[t]]])[0, 1]
        p_base[t] = y.mean()
    return pd.Series(p_cal), pd.Series(p_base)


v_crec = (cx['dir_gap']  == 'sube').dropna()
v_infl = (cx['dir_infl'] == 'sube').dropna()

p_crec, _ = calibrar(hacer_score(sig, CRE), v_crec)
p_infl, _ = calibrar(hacer_score({k: cand_i[k].dropna()
                                  for k in TRES_I}, TRES_I),
                     v_infl)

PP = pd.DataFrame({'crec': p_crec, 'infl': p_infl}).dropna()
PP.index = pd.to_datetime(PP.index) + MonthEnd(0)
PP.to_parquet(PROC + 'b7_p_ejes_calibradas.parquet')
# validado: (288, 2), 2001-11 a 2025-10


In [ ]:
# =====================================================================
# BLOQUE 5 — GATE
# =====================================================================
# P(fase) = producto de los ejes. Permiso empirico en D-25: el producto
# de los acuerdos por eje (69.7%) igualo la coincidencia exacta de fase
# (69.4%), o sea que los errores de los dos ejes son independientes.
#
# El gate viejo calibraba con la verdad hasta t-1, que NO existe: el
# fechado ex-post no esta disponible hasta 6-12 meses despues. Aca
# LAG=12 se aplica TAMBIEN al benchmark de tasa base, si no se compara
# una senal handicapeada contra un benchmark con superpoderes.
#
# Resultado, 253 meses 2004-10 a 2025-10:
#                      Brier   acierto  opuestos
#   senal completa    0.5926     53.0%     11.9%
#   solo crecimiento  0.7106     39.5%     17.8%
#   tasa base         0.7998     36.4%     20.6%
#   mejora de Brier sobre la tasa base: +25.9%   (viejo: +14.2%)
#   acierto por eje: crecimiento 70.0%, inflacion 71.1%
#
# Combinar los dos ejes hizo robusto lo que ninguno lograba solo:
# crecimiento aislado da +11% de mejora, los dos juntos +25.9%.
#
# La matriz de confusion muestra el sesgo de D-30 amplificado por el
# argmax: predice fases de crecimiento-subiendo el 85% del tiempo
# cuando la realidad es 64%. Deteccion: Overheat 84.9%, Recovery 52.9%,
# Reflation 23.1%, Stagflation 15.8%. NO SE USA EL ARGMAX en la
# cartera, y la ponderacion por probabilidad recupera lo que la
# etiqueta tira (ver bloque 9).

ORDEN = ['Recovery', 'Overheat', 'Stagflation', 'Reflation']
OPUESTO = {'Recovery': 'Stagflation', 'Stagflation': 'Recovery',
           'Overheat': 'Reflation',   'Reflation': 'Overheat'}

PFASE = pd.DataFrame({
    'Recovery':    PP.crec * (1 - PP.infl),
    'Overheat':    PP.crec * PP.infl,
    'Stagflation': (1 - PP.crec) * PP.infl,
    'Reflation':   (1 - PP.crec) * (1 - PP.infl)})[ORDEN]
PFASE.to_parquet(PROC + 'b7_p_fase.parquet')



In [ ]:
# =====================================================================
# BLOQUE 6 — CLASES DE ACTIVO
# =====================================================================
# 865 meses, 1953-05 a 2025-05. La restriccion es GS10, no commodities.
#            anual    vol
#   acciones  11.81   15.08
#   commodit   8.63   13.52
#   bonos      5.23    6.35
#   cash       4.06    0.88
#
# ACCIONES Y CASH: Ken French. Retorno total = Mkt-RF + RF.
# BONOS: sinteticos desde GS10 (FRED no publica TR de Tesoros con
#   historia larga). Duracion de Macaulay de un par bond a 10 anios,
#   madurez constante, sin convexidad (~3bp con variaciones mensuales).
#   VALIDADO CONTRA SEIS EPISODIOS: 1982 +30.59% (real ~+30%),
#   1994 -7.71% (~-8%), 2008 +17.60% (~+18-20%), 2013 -8.28% (~-8%),
#   2022 -16.31% (real ~-17%).
# COMMODITIES: AQR "Commodities for the Long Run" (Levine, Ooi,
#   Richardson, Sasseville, FAJ 2018). Indice equiponderado de futuros
#   desde 1877, con el ROLL ADENTRO y no estimado. Retorno total =
#   exceso + RF.
#   Desviacion vs Merrill, que usa GSCI: GSCI pondera por produccion y
#   esta ~60% energia, o sea que como "clase de activo" es en buena
#   medida una apuesta al petroleo. Para medir lo que la Table 5 afirma
#   sobre la clase, la equiponderacion es preferible.
# CREDITO: los OAS de ICE BofA son INVIABLES — FRED entrega solo una
#   ventana movil de 3 anios de las series licenciadas. Se reemplazan
#   por spreads de Moody's (BAA, AAA desde 1919).

import pandas_datareader.data as web

ff = web.DataReader('F-F_Research_Data_Factors',
                    'famafrench', start='1926-01')[0]
ff.index = ff.index.to_timestamp('M')
ff = ff / 100.0

SER = ['GS10', 'GS2', 'TB3MS', 'BAA', 'AAA', 'BAA10Y']
MAC = pd.DataFrame({s: a_mensual(fred.get_series(s))
                    for s in SER})


def bono_tr(serie, n):
    """Retorno total de un bono par de madurez constante."""
    y  = (serie / 100.0).dropna()
    D  = (1 + y) / y * (1 - (1 + y) ** (-n))
    Dm = D / (1 + y)
    return (y.shift(1) / 12 - Dm.shift(1) * y.diff()).dropna()


b10, b02 = bono_tr(MAC['GS10'], 10), bono_tr(MAC['GS2'], 2)

URL_AQR = ('https://www.aqr.com/-/media/AQR/Documents/Insights/'
           'Data-Sets/Commodities-for-the-Long-Run-Index-Level-'
           'Data-Monthly.xlsx')
DEST = RAW + 'aqr_commodities.xlsx'
if not os.path.exists(DEST):
    r = requests.get(URL_AQR, headers={'User-Agent': 'Mozilla/5.0'},
                     timeout=60)
    open(DEST, 'wb').write(r.content)

COLS_AQR = ['fecha', 'exc', 'exc_spot', 'carry_aj', 'spot', 'carry',
            'ls_exc', 'ls_exc_spot', 'ls_carry_aj', 'bc_agg',
            'bc_est', 'infl_est']
cmd = pd.read_excel(DEST, sheet_name='Commodities for the Long Run',
                    skiprows=10)
cmd.columns = COLS_AQR
cmd['fecha'] = pd.to_datetime(cmd.fecha)
cmd = cmd.dropna(subset=['fecha']).set_index('fecha').sort_index()
cmd.index = cmd.index + MonthEnd(0)
# ANOMALIA DOCUMENTADA: exc != exc_spot + carry_aj. Error medio
# 0.11pp mensual, sesgado en -0.10pp (~1.2pp anuales). La version
# multiplicativa solo lo reduce 25%. La hoja de definiciones no trae el
# detalle de calculo. NO SE USA LA DESCOMPOSICION en ningun calculo.
#
# El carry por decada si confirma los tres regimenes conocidos:
# +3 a +14pp de 1900 a 1990, -2.1 y -3.1 en 2000 y 2010 (contango
# post-financiarizacion), +1.1 en 2020 (vuelve la backwardation).

AC = pd.DataFrame({
    'acciones':    ff['Mkt-RF'] + ff['RF'],
    'bonos':       b10,
    'commodities': cmd['exc'] + ff['RF'],
    'cash':        ff['RF'],
}).dropna()


In [ ]:
# =====================================================================
# BLOQUE 7 — BOOTSTRAP POR EPISODIO
# =====================================================================
# La unidad de observacion es el EPISODIO, no el mes. Dos meses
# consecutivos de Stagflation no son dos observaciones: son un mismo
# shock. Remuestrear meses infla la significancia.
#
# POR EJE, 865 meses, 18-22 episodios por lado. SOBREVIVEN 4 DE 6 Y
# NO HAY NINGUNA INVERSION DE SIGNO:
#               crecimiento                 inflacion
#   acciones    +15.86 [ +9.82, +21.56]*   -10.38 [-17.31,  -3.68]*
#   bonos        -1.00 [ -4.77,  +2.58]     -6.46 [-10.04,  -2.86]*
#   commodities  +2.53 [ -4.79, +10.04]    +14.48 [ +7.16, +22.97]*
#
# Commodities NO responde al crecimiento y responde fuerte a la
# inflacion. Los bonos tampoco responden al crecimiento: son cobertura
# de inflacion invertida, no apuesta al ciclo.
#
# Medido por FASE, Stagflation x commodities apenas sobrevive (IC +0.89
# a +23.90, rango de 23 puntos). Como relacion de EJE queda solido. El
# efecto era real y estaba mal atribuido — resuelve la sospecha de que
# fueran solo los shocks del 73 y el 79.
#
# REPLICA DE LA TABLE 5 (ganador por fase, exceso sobre cash):
#   Recovery    acciones    +19.55  ✓ Merrill
#   Overheat    commodities +11.55  ✓ Merrill
#   Reflation   bonos        +6.64  ✓ Merrill
#   Stagflation commodities +11.69  ✗ (Merrill dice cash)
# En Stagflation acciones (-9.96) y bonos (-4.02) son negativos, asi
# que cash SI le gana a los dos. Lo que el paper no captura es que
# commodities les gana a todos. El modelo de ejes lo reproduce sin
# imponerlo: en la esquina (-,+) acciones y bonos se recortan a cero,
# commodities queda positivo, y el remanente va a cash.

CLA = ['acciones', 'bonos', 'commodities']
EXC = AC.sub(AC['cash'], axis=0)[CLA]


def episodios(lab):
    g = (lab != lab.shift()).cumsum()
    return [x.index for _, x in lab.groupby(g)]


def _boot(sumas, largos, B, rng):
    k = len(sumas)
    p = rng.integers(0, k, (B, k))
    return sumas[p].sum(1) / largos[p].sum(1)


def boot_eje(x, dir_, B=5000, seed=SEED):
    """media(sube) - media(baja), remuestreando cada lado aparte."""
    ep = episodios(dir_)
    lados = {}
    for lado in ['sube', 'baja']:
        v = [x.loc[i].values for i in ep
             if dir_.loc[i[0]] == lado]
        lados[lado] = (np.array([a.sum() for a in v], float),
                       np.array([len(a) for a in v], float))
    (su, nu), (sb, nb) = lados['sube'], lados['baja']
    rng = np.random.default_rng(seed)
    d = (_boot(su, nu, B, rng) - _boot(sb, nb, B, rng)) * 1200
    return (su.sum() / nu.sum() - sb.sum() / nb.sum()) * 1200, \
           np.percentile(d, [2.5, 97.5]), (d > 0).mean()


In [ ]:
# =====================================================================
# BLOQUE 8 — NIVEL 1: CLASES DE ACTIVO
# =====================================================================
#   score_c = alpha_c + Bg(c)·zg + Bi(c)·zi
#
# alpha es el exceso INCONDICIONAL sobre cash en ventana expansiva.
# Sin el, el modelo solo precia el diferencial condicional y tira el
# premio de riesgo: la version sin alpha se perdia quince anios de
# mercado alcista (2009 -0.1% contra +35.3% de acciones) porque las
# acciones quedaban en 0% casi siempre.
#
# alpha encaja con el centrado 'exp', que tiene media cero por
# construccion: alpha captura el nivel, z el desvio, y no se pisan. Con
# 'medio' habria doble conteo porque zg promedia +0.44.
#
# Coeficientes expansivos al final de la muestra vs bootstrap de
# muestra completa — el control de que el rezago esta bien alineado:
#   acciones    +16.23 / -10.26   vs   +15.86 / -10.38
#   bonos        -1.06 /  -6.55   vs    -1.00 /  -6.46
#   commodities  +2.28 / +14.62   vs    +2.53 / +14.48
#
# RESULTADO, 2003-10 a 2025-05:
#                    anual   vol     SR   maxDD  peor12  cash%  rot
#   exp/abs/alpha     8.36  9.50   0.88  -17.2   -15.9   14.0   190
#   exp/abs/sin       7.38  6.87   1.07  -11.3    -3.2   45.3   225
#   exp/pos/alpha     8.52 11.84   0.72  -36.2   -35.5    0.0   175
#   60/40             7.50  9.27   0.81  -28.6   -24.6
#   100% acciones    10.45 15.37   0.68  -50.3   -42.6
#   equipeso 4        5.38  6.28   0.86  -22.8   -22.7
#
# ESPECIFICACION PRINCIPAL: exp/abs/alpha, elegida por razon TEORICA
# pre-registrada (el retorno esperado es nivel mas desvio; omitir el
# nivel es error de especificacion) y no por Sharpe. El costo se
# reporta: el Sharpe baja de 1.07 a 0.88.
#
# HONESTIDAD SOBRE LA SIGNIFICANCIA:
#   exceso vs 60/40  +0.50 pp   IC95 [-3.74, +4.49]   P>0 0.592
# La ventaja de retorno NO esta establecida. Lo que si se sostiene es
# el drawdown (-17.2 vs -28.6), concentrado y direccional: cash en 2008,
# commodities en 2022.
#
# BRECHA CONTRA EL RELOJ PERFECTO:
#   real-time  SR 0.88   rot 190%
#   ex-post    SR 1.13   rot  59%
# Capturamos 78% del Sharpe alcanzable — mejor que el 55% del IR de la
# capa de acciones. Y el techo esta cerca: incluso con reloj perfecto
# el Sharpe es 1.13. EL CUELLO DE BOTELLA NO ES LA SENAL.

ALPHA = {cl: EXC[cl].expanding().mean().shift(LAG) for cl in CLA}
BET = {}
for cl in CLA:
    for k, d_ in [('g', v_crec), ('i', v_infl)]:
        dd = d_.reindex(EXC.index).map({True: 'sube', False: 'baja'})
        su = EXC[cl].where(dd == 'sube').expanding().mean()
        ba = EXC[cl].where(dd == 'baja').expanding().mean()
        BET[(cl, k)] = (su - ba).shift(LAG)


def construir(centrado='exp', norma='abs', usar_alpha=True):
    ix = PP.index.intersection(EXC.index)
    pc, pi = PP.crec.reindex(ix), PP.infl.reindex(ix)
    if centrado == 'medio':
        zg, zi = 2 * (pc - 0.5), 2 * (pi - 0.5)
    else:
        zg = 2 * (pc - pc.expanding(min_periods=24).mean())
        zi = 2 * (pi - pi.expanding(min_periods=24).mean())
    S = pd.DataFrame(index=ix, columns=CLA, dtype=float)
    for cl in CLA:
        a = ALPHA[cl].reindex(ix) if usar_alpha else 0.0
        S[cl] = (a + BET[(cl, 'g')].reindex(ix) * zg
                   + BET[(cl, 'i')].reindex(ix) * zi)
    S = S.dropna()
    pos = S.clip(lower=0)
    # 'abs': un score negativo REDUCE el riesgo total en vez de
    # empujar hacia las otras clases. Reproduce la lectura de Merrill
    # para Stagflation (cash manda, commodities de agregado).
    den = pos.sum(1) if norma == 'pos' else S.abs().sum(1)
    W = pos.div(den.where(den > 0), axis=0).fillna(0)
    W['cash'] = (1 - W[CLA].sum(1)).clip(lower=0)
    return W


W1 = construir('exp', 'abs', True)



In [ ]:
# =====================================================================
# BLOQUE 9 — NIVEL 2 Y 3: LA PATA DE RENTA VARIABLE
# =====================================================================
# Panel de factores del pipeline 05. Ya viene filtrado al universo
# point-in-time (el merge con f2_universo_pit conserva el 100% de las
# filas) y trae mc_live y fecha_disp, o sea las correcciones D-16/D-17.
#
# ATENCION A LAS UNIDADES: ret_total viene en PORCENTAJE, no en
# fraccion. AC (bloque 6) viene en fraccion. Al componer el arbol hay
# que dividir por 100.
#
# RECONCILIACION PARCIAL, declarada: no se reproduce exactamente el
# L0=8.65 / L2b=11.66 de fase5_decisiones.md. Lo mas cerca fue 8.91
# sobre 334 meses. La evidencia de que es el mismo panel es fuerte
# (vol 16.14 vs 16.03, spread 2.76 vs 3.01, mismas correcciones) pero
# la diferencia de 0.26pp no se explico. NO SE PERSIGUIO MAS porque
# aquellos numeros se calcularon con la senal macro SIN CALIBRAR y
# sobre otra ventana: hay que recalcularlos igual. Los de aca los
# reemplazan.
#
# EXTREMOS VERIFICADOS, no winsorizados: los mayores fwd1 son eventos
# reales (GME +1625% en ene-2021, Tupperware +434%, Regeneron +350%,
# Freddie Mac y Prologis en el rebote de 2009), no errores de split.
# Recortar en +-200% no mueve el indice; en +-100% lo mueve 0.03pp.

COLS_F = ['mes', 'ric', 'trbc', 'mc_live',
          'quality', 'growth', 'momentum']
PF = V2 + '/data/processed/f2_factores.parquet'
d = pd.read_parquet(PF, columns=COLS_F)

ret = pd.read_parquet(V2 + '/data/processed/f2_retornos_m.parquet',
                      columns=['ric', 'mes', 'ret_total'])
ret = ret.sort_values(['ric', 'mes'])
ret['fwd1'] = ret.groupby('ric')['ret_total'].shift(-1)
# el shift solo vale si el mes siguiente es el contiguo. Sin esto, una
# accion que falta un mes aparea el retorno de dos meses despues, y el
# error va SIEMPRE en la misma direccion: infla a los nombres que
# dejaron de cotizar, que son los que se derrumbaron.
_sig = ret.groupby('ric')['mes'].shift(-1)
ret.loc[~(_sig - ret.mes).dt.days.between(26, 35), 'fwd1'] = np.nan

d = d.merge(ret[['ric', 'mes', 'fwd1']], on=['ric', 'mes'], how='left')
d['mc_live'] = pd.to_numeric(d.mc_live, errors='coerce')


def patas_q(df, f='quality', top=0.2, min_sec=10):
    """Quintil superior del factor DENTRO de cada sector, cap-weighted.
    La neutralidad sectorial es obligada: si la seleccion pudiera
    concentrarse libremente elegiria sectores por su nivel medio del
    factor y contaminaria la inclinacion macro, que es lo que queremos
    medir aparte."""
    s = df.dropna(subset=[f, 'fwd1', 'trbc', 'mc_live'])
    s = s[s.mc_live > 0].copy()
    n = s.groupby(['mes', 'trbc'])[f].transform('size')
    s = s[n >= min_sec]
    s['pr'] = s.groupby(['mes', 'trbc'])[f].rank(pct=True)
    sel = s[s.pr > 1 - top]
    return (sel.groupby(['mes', 'trbc'])
              .apply(lambda g: (g.fwd1 * g.mc_live).sum()
                               / g.mc_live.sum(), include_groups=False)
              .unstack())


_b = d.dropna(subset=['fwd1', 'mc_live', 'trbc'])
_b = _b[_b.mc_live > 0]
wm = (_b.groupby(['mes', 'trbc']).mc_live.sum().unstack().fillna(0))
wm = wm.div(wm.sum(1), axis=0)
rel = (_b.groupby(['mes', 'trbc'])
         .apply(lambda g: (g.fwd1 * g.mc_live).sum() / g.mc_live.sum(),
                include_groups=False).unstack())
SEC = list(rel.columns)

# Sin seleccion posible (<10 nombres en el sector) la cartera se queda
# con el SECTOR ENTERO. Excluirlo cambiaria implicitamente los pesos
# sectoriales y mezclaria seleccion con asignacion.
PQ = patas_q(d).reindex(columns=SEC).fillna(rel)

# --- inclinacion sectorial: los 4 sectores del bootstrap de F4 ---
ESCALA = 0.25
SURV = {'Energy':                 [('infl', +1)],
        'Consumer Cyclicals':     [('infl', -1), ('crec', -1)],
        'Utilities':              [('crec', -1)],
        'Consumer Non-Cyclicals': [('crec', -1)]}

ixa = (PQ.index.intersection(PP.index)
         .intersection(wm.index).sort_values())
pp_ = PP.reindex(ixa)
w0 = wm.reindex(ixa)[SEC]
R_, Q_ = rel.reindex(ixa)[SEC], PQ.reindex(ixa)[SEC]


def tilt(centrado='exp'):
    """Inclinacion autofinanciada contra el indice, proporcional a los
    pesos de mercado (D-26)."""
    z = (pp_ - 0.5 if centrado == 'medio'
         else pp_ - pp_.expanding(min_periods=24).mean())
    T = pd.DataFrame(0.0, index=ixa, columns=SEC)
    for sec, lst in SURV.items():
        for eje, sg in lst:
            T[sec] += ESCALA * sg * z[eje]
    return T - w0.mul(T.sum(1), axis=0)


# RESULTADO, 290 meses 2001-11 a 2025-12 (todo en PORCENTAJE):
#                anual    vol     SR
#   L0 indice     9.49  15.35   0.62
#   L2b factor   11.24  13.99   0.80
#   L2 medio     11.93  14.39   0.83
#   L2 exp       11.77  13.42   0.88
#
# Se adopta 'exp', misma convencion que el nivel 1 y por la misma
# razon (D-30): con P(crec) sesgada 8pp alta, (P-0.5) deja una
# inclinacion permanente.
#
# LA VENTANA CUESTA: la capa aporta +2.28pp sobre el indice, no los
# +4.41 documentados en fase5_decisiones.md, y el factor solo +1.75 en
# vez de +3.01. No es un error: la ventana arranca en 2001-11 y se
# pierde 1997-2001, donde quality brillo en el pinchazo de las
# dot-com. Es el precio del warmup de la calibracion.
RV = ((w0 + tilt('exp')) * Q_).sum(1) / 100.0     # a FRACCION


In [ ]:
# =====================================================================
# BLOQUE 10 — COMPOSICION DEL ARBOL
# =====================================================================
# Los pesos salen del modelo macro sobre la CLASE; la pata se
# implementa con la ESTRATEGIA. Asi trabaja un asignador real: la
# vision es sobre el activo, la implementacion es activa.
#
# CONTROL DE ALINEACION (critico): RV ya es un retorno adelantado
# —se construyo con fwd1— asi que va SIN shift contra fwd. Medido:
#   corr(RV, mercado adelantado) = 0.934
#   corr con RV corrido un mes   = -0.003
# Si estuviera corrido un mes el resultado se veria plausible pero
# estaria midiendo la capacidad de predecir el pasado.
#
# TRES PATAS, 259 meses 2003-10 a 2025-04:
#   a) mercado French   8.44  vol 9.68  SR 0.87  maxDD -17.2
#   b) nuestro indice   8.23  vol 9.53  SR 0.86  maxDD -18.4
#   c) estrategia RV    8.41  vol 9.33  SR 0.90  maxDD -15.5
# (a) vs (b) no es el modelo: French rindio 11.25% y nuestro universo
# 10.25% en la ventana, o sea que el 8.44 estaba inflado por usar un
# indice de RV distinto del de la capa de acciones. La comparacion
# valida es (b) vs (c).
#
# RETORNO Y APORTE POR CLASE en la ventana:
#            retorno   vol    peso    aporte
#   acciones  12.50  13.72   42.7%   +6.30 pp
#   commod     5.20  14.93   32.1%   +1.41 pp
#   bonos      2.85   6.23   13.0%   +0.47 pp
#   cash       1.57   0.54   12.2%   +0.23 pp
# Commodities aporta 1.41pp TENIENDO SOLO 5.20% de retorno propio:
# el modelo los tuvo cuando servian, no en promedio. Es el reloj
# trabajando.

W1 = construir('exp', 'abs', True)
fwd = AC.shift(-1)
ixt = (W1.index.intersection(RV.index)
         .intersection(fwd.dropna().index).sort_values())
W = W1.reindex(ixt)
F = fwd.reindex(ixt).copy()
F['acciones'] = RV.reindex(ixt)
RET = (W * F[W.columns]).sum(1).dropna()

# ESPECIFICACION FINAL: 8.41% anual, vol 9.33%, SR 0.90, maxDD -15.5%
# contra 60/40 de 7.50% / 9.27% / 0.81 / -28.6%.
# NOTA SOBRE EL NIVEL DE RETORNO: no es un techo, es la version SIN
# APALANCAR. Con 40% menos volatilidad que las acciones, llevada a
# vol equivalente rinde ~12.9% con drawdown de ~-25% en vez de -50%.
# El Sharpe de 0.90 contra 0.65 es la afirmacion del trabajo; el nivel
# de retorno es una decision de cuanto riesgo tomar.

In [ ]:
# =====================================================================
# BLOQUE 11 — LA REDUNDANCIA DEFENSIVA  (hallazgo)
# =====================================================================
# Con acciones pesando 42.7% y la estrategia superando al indice en
# +2.25pp, el arbol deberia ganar +0.96pp. Gano +0.18.
#
#   contribucion ingenua  +0.96 pp
#   contribucion real     +0.18 pp
#   covarianza            -0.79 pp     corr(w_acc, exceso) = -0.197
#
# Exceso de la estrategia por cuartil de peso en acciones:
#   Q1 (w= 7.5%)  +6.71 pp      Q3 (w=56.5%)  -1.61 pp
#   Q2 (w=38.2%)  +2.83 pp      Q4 (w=69.0%)  +1.02 pp
#
# MECANISMO: quality gana en los meses malos de acciones (flight to
# quality, el argumento de QMJ) y el reloj sale de acciones en los
# meses malos. Dos coberturas del mismo riesgo, pagadas dos veces.
# Es lo OPUESTO a la interaccion dentro de la capa de acciones
# (+0.42pp): alli se reforzaban porque operaban sobre dimensiones
# distintas; aca se pisan porque operan sobre la misma.
#
# NO ES UN PROBLEMA DE TIMING: capturamos 78% del Sharpe del reloj
# perfecto. Con reloj perfecto la redundancia seria PEOR, porque
# cortaria la exposicion antes.
#
# REMEDIO TESTEADO Y RECHAZADO — estimar el coeficiente de acciones
# sobre la pata real en vez del mercado (no circular: se uso L2b, sin
# inclinacion macro):
#   beta mercado + RV   8.41  SR 0.90      <- se queda
#   beta pata + RV      8.04  SR 0.87
#   beta pata + L2b     7.74  SR 0.86
# El peso en acciones sube (42.7% -> 44.4%) y el retorno BAJA. El
# problema no era el peso sino el MOMENTO: ningun ajuste de nivel
# captura una ventaja concentrada en meses puntuales de caida.
# La tercera fila ademas confirma que la inclinacion sectorial DENTRO
# de la pata suma: quitarla empeora.
#
# HALLAZGO LATERAL, y es grande: el beta de inflacion de las acciones
#   mercado French 1953-2025:  -10.38
#   mercado French 2003-2025:   -0.23
# Una de las cuatro relaciones centrales del Investment Clock se
# sostiene en 72 anios pero NO se verifica en los ultimos 22. NO se
# re-estimo sobre la ventana corta: seria ajustar al periodo que
# estamos evaluando, y desde adentro de la muestra no hay forma de
# distinguir cambio de regimen de ruido de 20 episodios.
#
# LA REDUNDANCIA ESTA GRADUADA POR EL FACTOR — mecanismo predicho
# ANTES de correr, y confirmado en el orden exacto:
#   factor       cov    exceso  arbol   vol    SR   maxDD
#   quality    -0.79   +2.25    8.41   9.33  0.90  -15.5
#   momentum   -0.58   +2.59    8.76   9.95  0.88  -18.4
#   growth     -0.41   +2.46    8.88  10.24  0.87  -16.0
#   compuesto  -0.84   +2.19    8.15   9.51  0.86  -16.4
#
# EL COMPUESTO FALLA, contra la especificacion pre-registrada del plan
# y contra la intuicion habitual de que combinar factores ayuda.
# Explicacion probable: promediar tres z-scores selecciona nombres
# moderadamente buenos en todo en vez de excelentes en algo, y si el
# pago esta en los extremos, el promedio lo diluye.
#
# DECISION: quality, por la regla fijada de antemano en el proyecto —
# "si se solapan, usar la version no seleccionada". Las diferencias de
# retorno y Sharpe entre los tres no son distinguibles en 259 meses;
# lo que SI esta establecido es el ordenamiento de la covarianza.
# Quality era el factor primario antes de este test y es el nucleo
# metodologico del TFM.
#
# REENCUADRE: la covarianza negativa no es solo un costo. Es doble
# proteccion, y aparece donde tiene que aparecer — quality da el mejor
# Sharpe Y el mejor drawdown del conjunto. El resultado correcto es:
#   combinar un overlay macro defensivo con un factor defensivo cuesta
#   -0.79pp anuales y compra 2.9 puntos de drawdown maximo, y la
#   eleccion del factor gradua el intercambio.

In [ ]:
def pesos_nombre(f='quality', top=0.2, min_sec=10):
    s = d.dropna(subset=[f,'fwd1','trbc','mc_live'])
    s = s[s.mc_live > 0].copy()
    n = s.groupby(['mes','trbc'])[f].transform('size')
    s = s[n >= min_sec]
    s['pr'] = s.groupby(['mes','trbc'])[f].rank(pct=True)
    sel = s[s.pr > 1 - top].copy()
    sel['ws'] = (sel.mc_live
                 / sel.groupby(['mes','trbc'])
                      .mc_live.transform('sum'))
    return sel[['mes','ric','trbc','ws']]

Wsec = (w0 + tilt('exp')).reindex(ixt)
sel = pesos_nombre()
sel = sel[sel.mes.isin(ixt)]
sel = sel.join(Wsec.stack().rename('wsec'),
               on=['mes','trbc'])
sel['w'] = (sel.ws * sel.wsec
            * sel.mes.map(W['acciones']))
PN = (sel.pivot_table(index='mes', columns='ric',
                      values='w', fill_value=0)
         .reindex(ixt).fillna(0))
print('nombres unicos:', PN.shape[1])
print('suma de pesos de acciones (control):')
print((PN.sum(1) - W['acciones']).abs().max())

rot_eq = 0.5 * PN.diff().abs().sum(1)
rot_ot = 0.5 * W[['bonos','commodities','cash']
                 ].diff().abs().sum(1)
print(f'rotacion acciones (nombres) '
      f'{rot_eq.mean()*12*100:5.0f}%/anio')
print(f'rotacion otras clases        '
      f'{rot_ot.mean()*12*100:5.0f}%/anio')

print(f'\n{"acc/otros bp":>14s} {"costo":>7s}'
      f' {"neto":>7s} {"SR":>6s}')
bruto = RET.mean()*1200
vol_ = RET.std()*np.sqrt(12)*100
for be, bo in [(0,0), (10,2), (20,3), (30,5), (50,10)]:
    c = (rot_eq*be/10000 + rot_ot*bo/10000
         ).reindex(RET.index)
    n = (RET - c).dropna()
    a = n.mean()*1200
    print(f'{be:6d}/{bo:<7d} '
          f'{bruto-a:7.2f} {a:7.2f}'
          f' {a/(n.std()*np.sqrt(12)*100):6.2f}')


nombres unicos: 573
suma de pesos de acciones (control):
0.004682429586349679
rotacion acciones (nombres)   113%/anio
rotacion otras clases          121%/anio

  acc/otros bp   costo    neto     SR
     0/0          0.00    8.41   0.90
    10/2          0.14    8.27   0.89
    20/3          0.26    8.15   0.87
    30/5          0.40    8.01   0.86
    50/10         0.68    7.72   0.83


In [ ]:
b40 = (fwd['acciones']*.6
       + fwd['bonos']*.4).reindex(RET.index)
dif = (RET - b40).dropna()
fs = cx['fase'].reindex(dif.index).ffill()
ep = episodios(fs)
v = [dif.loc[i].values for i in ep]
s_ = np.array([a.sum() for a in v], float)
n_ = np.array([len(a) for a in v], float)
ms = _boot(s_, n_, 5000,
           np.random.default_rng(SEED)) * 1200
print(f'exceso vs 60/40 {dif.mean()*1200:+.2f} pp')
print(f'IC95 [{np.percentile(ms,2.5):+.2f},'
      f' {np.percentile(ms,97.5):+.2f}]'
      f'  P>0 {(ms>0).mean():.3f}  ep={len(ep)}')

exceso vs 60/40 +0.52 pp
IC95 [-3.61, +4.63]  P>0 0.593  ep=22


In [ ]:
mal = b40 < 0
dd_ = (RET - b40).reindex(b40.index)
print(f'meses malos de 60/40: {int(mal.sum())}'
      f' de {len(b40)}')
for nom, m in [('todos', pd.Series(True, b40.index)),
               ('60/40 negativo', mal)]:
    x = dd_[m].dropna()
    fs2 = cx['fase'].reindex(x.index).ffill()
    ep2 = episodios(fs2)
    v2 = [x.loc[i].values for i in ep2 if len(x.loc[i])]
    s2 = np.array([a.sum() for a in v2], float)
    n2 = np.array([len(a) for a in v2], float)
    b2 = _boot(s2, n2, 5000,
               np.random.default_rng(SEED)) * 100
    print(f'{nom:16s} exceso medio mensual '
          f'{x.mean()*100:+.2f}%  '
          f'IC95 [{np.percentile(b2,2.5):+.2f},'
          f' {np.percentile(b2,97.5):+.2f}]'
          f'  P>0 {(b2>0).mean():.3f}')

meses malos de 60/40: 87 de 259
todos            exceso medio mensual +0.04%  IC95 [-0.30, +0.39]  P>0 0.593
60/40 negativo   exceso medio mensual +0.87%  IC95 [+0.11, +1.72]  P>0 0.987


In [ ]:
import statsmodels.api as sm

x = b40.reindex(RET.index).dropna()
y = RET.reindex(x.index)
dn = (x < 0).astype(float)

X = sm.add_constant(pd.DataFrame({
    'mkt': x, 'mkt_dn': x * dn}))
m = sm.OLS(y, X).fit(cov_type='HAC',
                     cov_kwds={'maxlags': 6})
print(m.summary().tables[1])

b_up = m.params['mkt']
b_dn = m.params['mkt'] + m.params['mkt_dn']
print(f'\nbeta en subas  {b_up:.3f}')
print(f'beta en bajas  {b_dn:.3f}')
print(f'alpha anual    {m.params["const"]*1200:+.2f} pp'
      f'  (t={m.tvalues["const"]:.2f})')
print(f'convexidad     {m.params["mkt_dn"]:+.3f}'
      f'  (t={m.tvalues["mkt_dn"]:.2f})')

                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0020      0.002      0.959      0.337      -0.002       0.006
mkt            0.6983      0.142      4.913      0.000       0.420       0.977
mkt_dn        -0.0560      0.220     -0.255      0.799      -0.487       0.375

beta en subas  0.698
beta en bajas  0.642
alpha anual    +2.40 pp  (t=0.96)
convexidad     -0.056  (t=-0.25)
